# PFFT GPU Miner - Colab Edition
1. Runtime > Change runtime type > **T4 GPU**
2. Run All (Ctrl+F9)
3. Use different Google accounts for multi-GPU (1 account = 1 GPU)

In [ ]:
# Cell 1: Setup
!pip install -q web3 eth-account
!apt-get install -qq build-essential > /dev/null 2>&1
!nvidia-smi

In [ ]:
# Cell 2: CUDA Miner Compile
%%writefile pfft_miner.cu
#include <stdio.h>
#include <stdint.h>
#include <string.h>
#include <cuda_runtime.h>

__device__ __constant__ uint64_t RC[24] = {
    0x0000000000000001ULL, 0x0000000000008082ULL, 0x800000000000808aULL,
    0x8000000080008000ULL, 0x000000000000808bULL, 0x0000000080000001ULL,
    0x8000000080008081ULL, 0x8000000000008009ULL, 0x000000000000008aULL,
    0x0000000000000088ULL, 0x0000000080008009ULL, 0x000000008000000aULL,
    0x000000008000808bULL, 0x800000000000008bULL, 0x8000000000008089ULL,
    0x8000000000008003ULL, 0x8000000000008002ULL, 0x8000000000000080ULL,
    0x000000000000800aULL, 0x800000008000000aULL, 0x8000000080008081ULL,
    0x8000000000008080ULL, 0x0000000080000001ULL, 0x8000000080008008ULL
};

__device__ void keccak256_preimage(const uint8_t* input, int inlen, uint8_t* output) {
    uint64_t state[25] = {0};
    int rate = 136;
    int offset = 0;
    while (offset < inlen) {
        int block_size = (inlen - offset < rate) ? (inlen - offset) : rate;
        for (int i = 0; i < block_size; i++)
            ((uint8_t*)state)[i] ^= input[offset + i];
        offset += block_size;
        if (block_size == rate) {
            for (int round = 0; round < 24; round++) {
                uint64_t C[5], D[5];
                for (int x = 0; x < 5; x++)
                    C[x] = state[x] ^ state[x+5] ^ state[x+10] ^ state[x+15] ^ state[x+20];
                for (int x = 0; x < 5; x++) {
                    D[x] = C[(x+4)%5] ^ ((C[(x+1)%5] << 1) | (C[(x+1)%5] >> 63));
                    for (int y = 0; y < 25; y += 5) state[y+x] ^= D[x];
                }
                uint64_t t = state[1], tmp;
                static const int piln[24] = {10,7,11,17,18,3,5,16,8,21,24,4,15,23,19,13,12,2,20,14,22,9,6,1};
                static const int rotc[24] = {1,3,6,10,15,21,28,36,45,55,2,14,27,41,56,8,25,43,62,18,39,61,20,44};
                for (int i = 0; i < 24; i++) {
                    int j = piln[i]; tmp = state[j];
                    state[j] = (t << rotc[i]) | (t >> (64 - rotc[i])); t = tmp;
                }
                for (int y = 0; y < 25; y += 5) {
                    uint64_t t0=state[y], t1=state[y+1], t2=state[y+2], t3=state[y+3], t4=state[y+4];
                    state[y]=t0^((~t1)&t2); state[y+1]=t1^((~t2)&t3);
                    state[y+2]=t2^((~t3)&t4); state[y+3]=t3^((~t4)&t0); state[y+4]=t4^((~t0)&t1);
                }
                state[0] ^= RC[round];
            }
        }
    }
    ((uint8_t*)state)[inlen % rate] ^= 0x01;
    ((uint8_t*)state)[rate - 1] ^= 0x80;
    for (int round = 0; round < 24; round++) {
        uint64_t C[5], D[5];
        for (int x = 0; x < 5; x++)
            C[x] = state[x] ^ state[x+5] ^ state[x+10] ^ state[x+15] ^ state[x+20];
        for (int x = 0; x < 5; x++) {
            D[x] = C[(x+4)%5] ^ ((C[(x+1)%5] << 1) | (C[(x+1)%5] >> 63));
            for (int y = 0; y < 25; y += 5) state[y+x] ^= D[x];
        }
        uint64_t t = state[1], tmp;
        static const int piln[24] = {10,7,11,17,18,3,5,16,8,21,24,4,15,23,19,13,12,2,20,14,22,9,6,1};
        static const int rotc[24] = {1,3,6,10,15,21,28,36,45,55,2,14,27,41,56,8,25,43,62,18,39,61,20,44};
        for (int i = 0; i < 24; i++) {
            int j = piln[i]; tmp = state[j];
            state[j] = (t << rotc[i]) | (t >> (64 - rotc[i])); t = tmp;
        }
        for (int y = 0; y < 25; y += 5) {
            uint64_t t0=state[y], t1=state[y+1], t2=state[y+2], t3=state[y+3], t4=state[y+4];
            state[y]=t0^((~t1)&t2); state[y+1]=t1^((~t2)&t3);
            state[y+2]=t2^((~t3)&t4); state[y+3]=t3^((~t4)&t0); state[y+4]=t4^((~t0)&t1);
        }
        state[0] ^= RC[round];
    }
    memcpy(output, state, 32);
}

__global__ void mine_kernel(uint8_t* challenge, uint64_t start_nonce, uint64_t* result_nonce, int* found, uint8_t* target) {
    uint64_t tid = blockIdx.x * blockDim.x + threadIdx.x;
    uint64_t nonce = start_nonce + tid;
    if (*found) return;
    uint8_t input[64];
    memcpy(input, challenge, 32);
    memset(input + 32, 0, 24);
    for (int i = 0; i < 8; i++)
        input[56 + i] = (nonce >> (56 - i*8)) & 0xFF;
    uint8_t hash[32];
    keccak256_preimage(input, 64, hash);
    bool valid = false;
    for (int i = 0; i < 32; i++) {
        if (hash[i] < target[i]) { valid = true; break; }
        if (hash[i] > target[i]) { break; }
    }
    if (valid) { atomicExch(found, 1); *result_nonce = nonce; }
}

int main(int argc, char** argv) {
    if (argc < 4) { printf("Usage: %s <challenge> <target> <start_nonce>\n", argv[0]); return 1; }
    uint8_t challenge[32], target[32];
    const char* ch_hex = argv[1] + (argv[1][0]=='0' && argv[1][1]=='x' ? 2 : 0);
    const char* tg_hex = argv[2] + (argv[2][0]=='0' && argv[2][1]=='x' ? 2 : 0);
    for (int i = 0; i < 32; i++) {
        sscanf(ch_hex + i*2, "%2hhx", &challenge[i]);
        sscanf(tg_hex + i*2, "%2hhx", &target[i]);
    }
    uint64_t start_nonce = strtoull(argv[3], NULL, 10);
    uint8_t *d_challenge, *d_target; uint64_t *d_result; int *d_found;
    cudaMalloc(&d_challenge, 32); cudaMalloc(&d_target, 32);
    cudaMalloc(&d_result, 8); cudaMalloc(&d_found, 4);
    cudaMemcpy(d_challenge, challenge, 32, cudaMemcpyHostToDevice);
    cudaMemcpy(d_target, target, 32, cudaMemcpyHostToDevice);
    int threads = 256, blocks = 4096;
    uint64_t batch = (uint64_t)threads * blocks;
    uint64_t nonce = start_nonce; int found_h = 0; uint64_t result_h = 0;
    while (!found_h) {
        cudaMemset(d_found, 0, 4);
        mine_kernel<<<blocks, threads>>>(d_challenge, nonce, d_result, d_found, d_target);
        cudaDeviceSynchronize();
        cudaMemcpy(&found_h, d_found, 4, cudaMemcpyDeviceToHost);
        if (found_h) cudaMemcpy(&result_h, d_result, 8, cudaMemcpyDeviceToHost);
        nonce += batch;
    }
    printf("%llu\n", result_h);
    cudaFree(d_challenge); cudaFree(d_target); cudaFree(d_result); cudaFree(d_found);
    return 0;
}

In [ ]:
# Cell 3: Compile CUDA miner for T4 (sm_75)
!nvcc -O3 -arch=sm_75 -o pfft_miner pfft_miner.cu
!ls -la pfft_miner

In [ ]:
# Cell 4: Mining Loop
import subprocess, time, os, random
from web3 import Web3
from eth_account import Account

# ===============================================================
# CONFIG - ISI PRIVATE KEY DI BAWAH INI
# Kalau kosong, otomatis generate wallet baru
# ===============================================================
RPC_URL = "https://rpc.ankr.com/eth/YOUR_ANKR_KEY_HERE"
PRIVATE_KEY = ""  # <-- ISI PK DISINI (contoh: 0xabc123...)
# ===============================================================

if PRIVATE_KEY:
    acct = Account.from_key(PRIVATE_KEY)
else:
    acct = Account.create()
    PRIVATE_KEY = acct.key.hex()

WALLET = acct.address
print(f"\n{'='*60}")
print(f"WALLET: {WALLET}")
print(f"PK: {PRIVATE_KEY}")
print(f"Send 0.003 ETH to {WALLET} then mining starts")
print(f"{'='*60}\n")

CONTRACT = "0xEFAd2Eab7172dDEbE5Ce7a41f5Ddf8fCcE4Ca0CB"
ABI = [
    {"inputs":[{"name":"user","type":"address"}],"name":"currentPowChallenge","outputs":[{"name":"","type":"bytes32"}],"stateMutability":"view","type":"function"},
    {"inputs":[],"name":"POW_TARGET","outputs":[{"name":"","type":"uint256"}],"stateMutability":"view","type":"function"},
    {"inputs":[{"name":"powNonce","type":"uint256"}],"name":"freeMint","outputs":[],"stateMutability":"nonpayable","type":"function"},
    {"inputs":[{"name":"account","type":"address"}],"name":"balanceOf","outputs":[{"name":"","type":"uint256"}],"stateMutability":"view","type":"function"},
    {"inputs":[],"name":"totalSupply","outputs":[{"name":"","type":"uint256"}],"stateMutability":"view","type":"function"}
]

w3 = Web3(Web3.HTTPProvider(RPC_URL))
contract = w3.eth.contract(address=CONTRACT, abi=ABI)

# Wait for ETH
print("Waiting for ETH...")
while True:
    bal = w3.eth.get_balance(WALLET)
    if bal > w3.to_wei(0.0005, 'ether'):
        print(f"Balance: {w3.from_wei(bal, 'ether')} ETH - Starting miner!")
        break
    time.sleep(5)

# Mining loop
total_minted = 0
mints = 0
start_time = time.time()

while True:
    try:
        challenge = contract.functions.currentPowChallenge(WALLET).call()
        target = contract.functions.POW_TARGET().call()

        challenge_hex = '0x' + challenge.hex()
        target_hex = '0x' + hex(target)[2:].zfill(64)
        start_nonce = random.randint(0, 2**48)

        print(f"\n[{time.strftime('%H:%M:%S')}] Round {mints+1} - challenge: {challenge_hex[:18]}...")
        print(f"  Target: {target_hex[:18]}...")

        # Run CUDA miner
        t0 = time.time()
        result = subprocess.run(
            ['./pfft_miner', challenge_hex, target_hex, str(start_nonce)],
            capture_output=True, text=True, timeout=600
        )
        solve_time = time.time() - t0

        if result.returncode != 0:
            print(f"  ERROR: {result.stderr[:200]}")
            time.sleep(5)
            continue

        nonce = int(result.stdout.strip())
        print(f"  ✓ Solved! nonce={nonce} time={solve_time:.1f}s")

        # Submit tx
        tx = contract.functions.freeMint(nonce).build_transaction({
            'from': WALLET,
            'nonce': w3.eth.get_transaction_count(WALLET),
            'gas': 300000,
            'gasPrice': w3.to_wei(0.2, 'gwei'),
            'chainId': 1
        })
        signed = w3.eth.account.sign_transaction(tx, PRIVATE_KEY)
        tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
        print(f"  Tx: {tx_hash.hex()}")

        # Wait for receipt
        receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=120)
        if receipt.status == 1:
            bal = contract.functions.balanceOf(WALLET).call()
            pfft_bal = bal / 1e18
            mints += 1
            elapsed = (time.time() - start_time) / 60
            print(f"  ✓ MINTED! Total: {pfft_bal:.1f} PFFT | {mints} mints | {elapsed:.1f}min")
            print(f"  https://etherscan.io/tx/0x{tx_hash.hex()}")
        else:
            print(f"  ✗ REVERTED! gasUsed={receipt.gasUsed}")
            time.sleep(2)

    except KeyboardInterrupt:
        print(f"\nStopped. Total: {mints} mints")
        break
    except Exception as e:
        print(f"  Error: {str(e)[:200]}")
        time.sleep(5)